In [2]:
import sqlite3

DB_PATH = "rechtspraak_cases.db"

conn = sqlite3.connect(DB_PATH)

# Check the number of tables in the database
cursor = conn.execute("SELECT COUNT(*) FROM cases")
print(cursor.fetchone()[0])


cursor = conn.execute("SELECT ecli, abstract FROM cases")


for ecli, abstract in cursor:
    print(f"ECLI: {ecli}")
    print(f"Abstract: {abstract}")
    print("-" * 80)

conn.close()

9788
ECLI: ECLI:NL:GHARL:2025:8014
Abstract: WWZ. Ontbinding vanwege verstoorde arbeidsverhouding. Geen sprake van ernstige verwijtbaarheid aan de kant van werkgever dus geen billijke vergoeding.
--------------------------------------------------------------------------------
ECLI: ECLI:NL:GHDHA:2025:2585
Abstract: bewijslevering "bijzonder geval"in de zin van art. 9.2 van de CAO VO. verlenging tijdelijke arbeidsovereenkomst voor bepaalde tijd niet geldig.
--------------------------------------------------------------------------------
ECLI: ECLI:NL:RBROT:2025:14828
Abstract: Arbeidsrecht. Ontslag op staande voet. Werkgever beschuldigt werknemer van koperdiefstal en ontslaat hem op staande voet. Ontslag vernietigd. Koperdiefstal staat niet vast.
--------------------------------------------------------------------------------
ECLI: ECLI:NL:RBSHE:2012:7589
Abstract: Ontbindingsverzoek werknemer op 7 mei 2012, nadat de werkgever op 12 april 2012 het UWV heeft verzocht om toestemming om de

In [4]:
METADATA_LINK = "https://data.rechtspraak.nl/uitspraken/content?id=ECLI:NL:RBROT:2025:14828&return=META"

In [8]:
import requests
from lxml import etree
from typing import Optional, Dict


NAMESPACES = {
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "dcterms": "http://purl.org/dc/terms/",
    "psi": "http://psi.rechtspraak.nl/",
}


def first_text(root, xpath: str) -> Optional[str]:
    nodes = root.xpath(xpath, namespaces=NAMESPACES)
    if not nodes:
        return None
    if isinstance(nodes[0], str):
        return nodes[0].strip()
    return (nodes[0].text or "").strip()


def extract_metadata(xml_bytes: bytes) -> Dict[str, Optional[str]]:
    root = etree.fromstring(xml_bytes)

    metadata = {
        # ECLI code
        "ecli": first_text(
            root,
            "//rdf:Description/dcterms:identifier[text()[starts-with(., 'ECLI:')]]/text()",
        ),

        # Rechtbank / instantie
        "rechtbank": first_text(
            root,
            "//rdf:Description/dcterms:creator/text()",
        ),

        # Publicatiedatum (from metadata description)
        "publicatiedatum": first_text(
            root,
            "//rdf:Description/dcterms:issued[@rdf:label='Publicatiedatum']/text()",
        ),

        # Abstract (korte inhoud)
        "abstract": first_text(
            root,
            "//rdf:Description/dcterms:abstract/text()",
        ),

        # HTML deeplink
        "link": first_text(
            root,
            "//rdf:Description[@rdf:about]/dcterms:identifier/text()",
        ),
    }

    return metadata     


<Element open-rechtspraak at 0x1119c8140>

In [1]:
from xml_to_md import RechtspraakConverter

METADATA_LINK = "https://data.rechtspraak.nl/uitspraken/content?id=ECLI:NL:RBROT:2025:14828"
resp = requests.get(METADATA_LINK, timeout=30)
xml_string = resp.content #etree.fromstring(resp.content)

converter = RechtspraakConverter(xml_string)
markdown = converter.to_markdown()
metadata = converter.extract_metadata()  # Returns dict
summary = converter.extract_summary()

NameError: name 'requests' is not defined

b'<?xml version="1.0" encoding="utf-8"?>\r\n<open-rechtspraak>\r\n  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:ecli="https://e-justice.europa.eu/ecli" xmlns:tr="http://tuchtrecht.overheid.nl/" xmlns:eu="http://publications.europa.eu/celex/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:bwb="bwb-dl" xmlns:cvdr="http://decentrale.regelgeving.overheid.nl/cvdr/" xmlns:psi="http://psi.rechtspraak.nl/" xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#">\r\n    <rdf:Description>\r\n      <dcterms:identifier>ECLI:NL:RBROT:2025:14828</dcterms:identifier>\r\n      <dcterms:format>text/xml</dcterms:format>\r\n      <dcterms:accessRights>public</dcterms:accessRights>\r\n      <dcterms:modified>2025-12-24T16:19:01</dcterms:modified>\r\n      <dcterms:issued rdfs:label="Publicatiedatum">2025-12-18</dcterms:issued>\r\n      <dcterms:publisher resourceIdentifier="http://rechtspraak.nl/">Raad voor de Rechtspraak</dcterms:publisher>\r\n      <dcterms:language>nl</dcterms:la

In [6]:
# Check classifications where is_contract_drafting_issue = false
conn = sqlite3.connect(DB_PATH)
cursor = conn.execute("""
    SELECT c.ecli, cl.is_contract_drafting_issue, c.abstract, cl.reason, cl.confidence
    FROM cases c
    JOIN classifications cl ON c.ecli = cl.ecli
    WHERE cl.is_contract_drafting_issue = 1
""")

for ecli, contract_related, abstract, reason, confidence in cursor:
    print(f"ECLI: {ecli}")
    print(f'Contract Related: {bool(contract_related)}')
    print(f"Abstract: {abstract}")
    print(f"Reason: {reason}")
    print(f"Confidence: {confidence}")
    print("-" * 80)

conn.close()

ECLI: ECLI:NL:GHSHE:2013:5478
Contract Related: True
Abstract: Ontslag wegens bedrijfseconomische omstandigheden. Uitleg Cao. Overgangsregeling. umulatie wachtgeld met ontslagvergoeding. Toepassing Cao-maatstaf.
Reason: Case involves interpretation of CAO (collective agreement) provisions regarding transition arrangements and compensation standards - contract interpretation issue
Confidence: 0.8
--------------------------------------------------------------------------------
ECLI: ECLI:NL:GHSHE:2013:1492
Contract Related: True
Abstract: Nakoming en uitleg van Sociaal Plan en daarmee verband houdende toezeggingen. Berekening van de daarin opgenomen vergoedingen.
Reason: Case involves interpretation ("uitleg") of Social Plan and related commitments, including calculation of compensations - this is contract interpretation
Confidence: 0.85
--------------------------------------------------------------------------------
ECLI: ECLI:NL:GHSHE:2014:1159
Contract Related: True
Abstract: uitleg l